# MICAI 2026 · MLOps Workshop
## Notebook 0 · Set Up Your Own MLflow Workspace (run once)
**Chihuahua, Mexico · November 2, 2026**

| | |
|---|---|
| **Workshop** | MICAI 2026 · MLOps Workshop: from raw data to a monitored model |
| **Dataset** | *Wine Quality* (white wine). P. Cortez, A. Cerdeira, F. Almeida, T. Matos and J. Reis (2009). *Modeling wine preferences by data mining from physicochemical properties.* Decision Support Systems 47(4), 547–553. Obtained from the UCI Machine Learning Repository. The file `modified.csv` is a deliberately corrupted copy prepared for this workshop. |
| **Tools** | Python, pandas, scikit-learn, SciPy, MLflow, Databricks and Google Colab |

**Declaration on the use of AI.** These materials were developed with the assistance of generative AI
(Anthropic's Claude, model Claude Opus 5.5), used to draft and review code, explanations and documentation.
The workshop instructors reviewed and validated all content and are responsible for it.

    ## Run this notebook once, before notebooks 1–3

    During the workshop the instructor prepared a shared MLflow workspace on Databricks. To run the notebooks on your
    own, you need **your own** Databricks MLflow workspace, prepared the same way. This notebook does it for you in
    about one minute:

    1. creates the folder `/Shared/MICAI_MLOps_2026` and the four MLflow experiments;
    2. publishes the **official dataset versions** (v1 drop, v2 impute and the fixed holdout) to
       `00_official_datasets`, copying the processed files stored in the repository;
    3. publishes the **reference ("production") model** to `02_training`, the model monitored in notebook 3.

    **Before you start**, follow steps 1–5 of the [setup guide](https://github.com/ITESM-MNA/MICAI_MLOps_2026/blob/main/docs/SETUP_YOUR_OWN_MLFLOW.md): create a free Databricks account, an
    access token, and the Colab Secret `MICAI_CONFIG` holding your config. Running this notebook again is safe: steps
    already done are skipped.

    **The notebooks**

    | # | Notebook | Question it answers | MLflow experiment |
|---|---|---|---|
| 0 | **➡ Set up your own workspace (once, on your own Databricks)** | Is everything ready? | `00_official_datasets` |
| 1 | Data quality, EDA and versioning | Can we trust the data? | `01_data_cleaning` |
| 2 | Training and experiment tracking | Which model, trained on which data version, works best? | `02_training` |
| 3 | Monitoring and data drift | Is the model still valid once the data changes? | `03_drift_monitoring` |

## Step 0 · Setup (about 1 minute)

Run the next cell. It has two fields:

1. **CONFIG_LINK**: where the config comes from (the Databricks address and your access key).
   - **On your own Databricks:** leave it **empty**. The cell reads your Colab Secret `MICAI_CONFIG`
     (key icon 🔑 on the left; switch on *Notebook access*). How to create it:
     [setup guide](https://github.com/ITESM-MNA/MICAI_MLOps_2026/blob/main/docs/SETUP_YOUR_OWN_MLFLOW.md).
   - **In a workshop:** paste the config link given by the instructor.
   - A file path also works, for example `/content/config.json` after uploading the file.
2. **NICKNAME**: any nickname used to tag your runs. Please don't use your real name.

The cell then, automatically:
- downloads the workshop code and data from GitHub into this Colab machine,
- installs MLflow,
- reads the config (the access key is never printed),
- connects this notebook to the MLflow experiment `00_official_datasets` on Databricks.

If anything fails (no config, a mistyped link, the network, an expired key), the cell says so in one line and
switches to a **local** MLflow store inside Colab. The notebook still runs end to end; your runs are just not saved
on Databricks.

> Each notebook runs on its own Colab machine, so this setup is repeated at the top of every notebook,
> with the same config and nickname.

In [ ]:
#@title Step 0 · Workshop setup (click ▶ to run)
CONFIG_LINK = ""  #@param {type:"string"}
NICKNAME = ""  #@param {type:"string"}

import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/ITESM-MNA/MICAI_MLOps_2026.git"
try:
    import google.colab  # noqa: F401  (only available inside Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = Path("/content/MICAI_MLOps_2026")
    if not REPO_DIR.exists():
        print("Downloading the workshop files...")
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL, str(REPO_DIR)], check=True)
    print("Installing MLflow (about 20 seconds)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements-colab.txt")], check=True)
else:  # running outside Colab, from the notebooks/ folder of the repository
    here = Path.cwd()
    REPO_DIR = Path(os.environ.get("MICAI_REPO_DIR", here.parent if (here.parent / "src").exists() else here))

sys.path.insert(0, str(REPO_DIR / "src"))
from micai_mlops.setup import setup_workshop
ws = setup_workshop(CONFIG_LINK, NICKNAME, notebook="00", repo_dir=REPO_DIR)

## 1. Prepare your workspace

The next cell checks that the repository copy of the official datasets matches the fingerprints in
`data/wine-quality/official_manifest.json`, publishes it to MLflow, and trains and publishes the reference model.
If the setup cell above could not connect to Databricks, it prepares the local MLflow store instead and says so:
check your config and run the setup cell again.

In [ ]:
from micai_mlops.workspace import prepare_workspace

result = prepare_workspace(ws)

## 2. Check the result

Open the links printed above. In Databricks, **Experiments** in the left sidebar now lists the four experiments
under `/Shared/MICAI_MLOps_2026`. In `00_official_datasets`, the run `official-datasets` shows the three datasets in
its **Datasets** column and the files under **Artifacts → official_datasets**.

**Next:** open notebook 1 and run it. In every notebook leave **CONFIG_LINK** empty: the setup cell reads your
Colab Secret.

> **Need to publish again?** Run `prepare_workspace(ws, force=True)`: it creates a new official run, which
> notebooks 2 and 3 then use (the most recent one is always picked).